## DM Project: Impact of Weather on US Flight Delays
This notebook fetches historical weather data from the Open-Meteo API
for the top 10 busiest US airports during June 2019-2023.
- API: Open-Meteo Historical Weather (https://open-meteo.com)
- No API key required
- Fields: temperature_max, precipitation_mm, windspeed_max_kmh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import requests
import time

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DM_Project/Data/flights_sample_3m.csv')
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], format='mixed', errors='coerce')
df = df.dropna(subset=['FL_DATE', 'ORIGIN'])

top_airports = df['ORIGIN'].value_counts().head(10).index.tolist()
print("Top 10 airports:", top_airports)

df_top = df[df['ORIGIN'].isin(top_airports)]
df_sample = df_top[df_top['FL_DATE'].dt.month == 6]
print("Sample shape:", df_sample.shape)

Top 10 airports: ['ATL', 'DFW', 'ORD', 'DEN', 'CLT', 'LAX', 'PHX', 'LAS', 'SEA', 'MCO']
Sample shape: (86302, 32)


In [ ]:
airport_coords = {
    'ATL': (33.6407, -84.4277),
    'DFW': (32.8998, -97.0403),
    'ORD': (41.9742, -87.9073),
    'DEN': (39.8561, -104.6737),
    'CLT': (35.2140, -80.9431),
    'LAX': (33.9425, -118.4081),
    'PHX': (33.4373, -112.0078),
    'LAS': (36.0840, -115.1537),
    'SEA': (47.4502, -122.3088),
    'MCO': (28.4312, -81.3081)
}
print("Airport coordinates ready!")

Airport coordinates ready!


In [ ]:
def get_weather(airport, date_str):
    lat, lon = airport_coords[airport]
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={date_str}&end_date={date_str}"
        f"&daily=temperature_2m_max,precipitation_sum,windspeed_10m_max"
        f"&timezone=America%2FNew_York"
    )
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        return {
            'ORIGIN': airport,
            'FL_DATE': date_str,
            'temperature_max': data['daily']['temperature_2m_max'][0],
            'precipitation_mm': data['daily']['precipitation_sum'][0],
            'windspeed_max_kmh': data['daily']['windspeed_10m_max'][0]
        }
    except:
        return None

print("Weather function ready!")

Weather function ready!


In [ ]:
unique_combos = df_sample[['FL_DATE', 'ORIGIN']].drop_duplicates()
unique_combos['FL_DATE'] = unique_combos['FL_DATE'].dt.strftime('%Y-%m-%d')

print("Total API calls to make:", len(unique_combos))

weather_records = []
total = len(unique_combos)

for i, row in enumerate(unique_combos.itertuples(), 1):
    result = get_weather(row.ORIGIN, row.FL_DATE)
    if result:
        weather_records.append(result)
    if i % 100 == 0:
        print(f"Progress: {i}/{total} ({round(i/total*100)}%)")
    time.sleep(0.1)

weather_df = pd.DataFrame(weather_records)
print("\nDone!")
print("Weather records fetched:", weather_df.shape)
print(weather_df.head())

Total API calls to make: 1500
Progress: 100/1500 (7%)
Progress: 200/1500 (13%)
Progress: 300/1500 (20%)
Progress: 400/1500 (27%)
Progress: 500/1500 (33%)
Progress: 600/1500 (40%)
Progress: 700/1500 (47%)
Progress: 800/1500 (53%)
Progress: 900/1500 (60%)
Progress: 1000/1500 (67%)
Progress: 1100/1500 (73%)
Progress: 1200/1500 (80%)
Progress: 1300/1500 (87%)
Progress: 1400/1500 (93%)
Progress: 1500/1500 (100%)

Done!
Weather records fetched: (1495, 5)
  ORIGIN     FL_DATE  temperature_max  precipitation_mm  windspeed_max_kmh
0    ATL  2021-06-11             27.5              10.4               19.3
1    ORD  2021-06-08             29.4               6.0                9.3
2    DFW  2021-06-01             24.4               0.9               19.4
3    SEA  2021-06-26             34.5               0.0               14.4
4    PHX  2022-06-10             44.5               0.0               20.8


In [ ]:
weather_df.to_csv('/content/drive/MyDrive/DM_Project/Data/weather_api_raw.csv', index=False)
print("weather_api_raw.csv saved to Drive!")

weather_api_raw.csv saved to Drive!
